# Первичный технический контроль данных эксперимента 2

**Статус:** машинный контроль выполнен; покомпонентные рекомендации остаются
кандидатными и имеют статус pending_manual_review.

Ноутбук устанавливает состав файлов эксперимента 2, выявляет дубликаты,
проверяет минимальные технические свойства записей и формирует отдельную
карточку для каждого измерения до разметки дыхания, ЭКГ и физической
интерпретации. Подтверждённый протокол серии приведён в
[паспорте эксперимента 2](10.00_Паспорт_эксперимента_2.md).

Исходные файлы и оба записанных канала сохраняются. Для записей Георга
90 и 100 мм действует поканальное решение: `BASE1/RHEO1` исключены из
количественных расчётов; `BASE2` остаётся условным кандидатом после проверки
задержек вдоха и выдоха; `RHEO2` — после ЭКГ-разметки и только по циклам без
предельных плато. Совместные расчёты каналов 1 и 2 для этих записей пока не
выполняются. Сами записи целиком не исключены.

Предыдущая смешанная редакция сохранена в
[архивном снимке 10.91](archive/legacy/10.91_Снимок_10.01_до_разделения_ответственности.ipynb).


## Область ответственности

Этот ноутбук отвечает за:

- инвентаризацию явно разрешённой области внешнего хранилища;
- проверку схемы CSV, числовых значений и временной шкалы;
- расчёт длительности и фактической частоты дискретизации;
- выявление побайтовых дубликатов по SHA-256;
- проверку формального наличия базового уровня каждого канала;
- поиск технических признаков длительных плато на крайних значениях RHEO;
- покомпонентные комментарии и поканальные ограничения количественного использования;
- формирование обезличенных кандидатных манифестов.

В этот ноутбук не входят: реконструкция дыхательных интервалов (11.01),
разметка ЭКГ (11.02), сравнение уровней на принятых вдохе и выдохе (12.01),
калибровка и объяснение аппаратных аномалий (10.02), а также расчёт кажущегося
удельного сопротивления (серия 33).

Результат 10.01 сообщает, какие записи и интервалы требуют ручной проверки и
как их предварительно передавать дальше. Он не подтверждает физиологическую
пригодность сигнала и не даёт оценок свойств тканей.


## Критерии машинной проверки

Проверяются ожидаемые столбцы TIME_s, ECG_V, BASE_1_Ω, RHEO_1_mΩ,
BASE_2_Ω, RHEO_2_mΩ, QS_1_Ω и QS_2_Ω, конечность чисел и строгое
возрастание времени.

Поле actual_active_channels означает только, что BASE превышал заданный порог
непрерывно в течение заданного времени. Оно не доказывает правильность
подключения, калибровку или наличие пригодного физиологического сигнала и
чувствительно к неизвестным знаку, масштабу и смещению BASE.

Дополнительный флаг плато срабатывает, если RHEO остаётся точно равным
минимальному или максимальному значению записи не менее 0,1 с. Порог 0,1 с —
техническая эвристика для ручного просмотра. Повторение одинаковых крайних
значений в разных файлах является признаком возможного ограничения шкалы или
цифрового насыщения, но не устанавливает причину и не даёт права исключить
канал целиком.

mtime используется только для проверки согласованности с порядком
140 → 50 мм. Оно не считается временем регистрации и могло измениться при
копировании.


In [1]:
# @title Инвентаризация файлов и формирование кандидатного манифеста
import datetime
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from record_qc import build_exp02_candidate_manifest

CONFIG_ENV = "KALMYKOV_EXP02_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному файлу "
        "config/exp02_paths.local.json"
    )

CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
SUBJECT_LABELS = {
    item["subject_id"]: item.get("report_label", item["subject_id"])
    for item in CONFIG["subjects"]
}

QC_MANIFEST_PATH, QC_MANIFEST = build_exp02_candidate_manifest(CONFIG_PATH)
RECORDS = QC_MANIFEST["records"]

role_labels = {
    "ttrkg_channel_1_with_side_channel_2_size_sweep": "размерный ряд",
    "excluded_duplicate": "побайтовый дубликат",
    "full_breathing_protocol_side_size_unknown": "полный протокол, размер неизвестен",
    "unclassified": "назначение не установлено",
}


def decision_label(item):
    if item["role"] == "excluded_duplicate":
        return "сохранён для прослеживаемости; не независимый повтор"
    if item.get("side_size_mm") is None:
        return "сохранён для протокольного анализа; размер неизвестен"
    if item["include"]:
        return "самостоятельная запись размерного ряда"
    return "сохранён; требуется отдельная ручная проверка"


rows = []
for item in RECORDS:
    file_qc = item["file_qc"]
    activity = file_qc["channel_activity"]
    source_path = (DATA_ROOT / item["relative_path"]).resolve()
    source_path.relative_to(DATA_ROOT)
    rows.append(
        {
            "доброволец": SUBJECT_LABELS[item["subject_id"]],
            "размер, мм": item.get("side_size_mm"),
            "роль": role_labels.get(item["role"], item["role"]),
            "решение": decision_label(item),
            "длительность, с": round(file_qc["duration_s"], 3),
            "частота, Гц": round(file_qc["sampling_frequency_hz"], 6),
            "каналы, прошедшие формальное правило": ", ".join(
                map(str, item["actual_active_channels"])
            ),
            "доля BASE1 выше порога": round(activity["1"]["active_fraction"], 4),
            "доля BASE2 выше порога": round(activity["2"]["active_fraction"], 4),
            "физических копий": item["copies_in_allowed_root"],
            "mtime": source_path.stat().st_mtime,
        }
    )

record_table = pd.DataFrame(rows)
display(
    record_table.drop(columns="mtime").sort_values(
        ["доброволец", "размер, мм"], na_position="last"
    ).reset_index(drop=True)
)

inventory_table = (
    pd.DataFrame.from_dict(QC_MANIFEST["inventory"], orient="index")
    .rename_axis("доброволец")
    .rename(
        columns={
            "csv_files_with_expected_schema": "CSV с ожидаемой схемой",
            "unique_content_records": "уникальных содержаний",
            "schema_mismatch_csv_count": "CSV с другой схемой",
            "unclassified_unique_records": "неклассифицированных уникальных",
        }
    )
)
display(inventory_table)

independent_size_records = [item for item in RECORDS if item["include"]]
other_retained_records = [item for item in RECORDS if not item["include"]]
activity_rule = RECORDS[0]["file_qc"]["channel_activity_rule"]

print("Статус манифеста: кандидатный, требуется ручная проверка.")
print("Независимых записей размерного ряда:", len(independent_size_records))
print("Других сохранённых записей:", len(other_retained_records))
print("Удалённых записей: 0; исключённых каналов: 0.")
print(
    "Формальное правило канала: BASE > "
    f"{activity_rule['base_threshold_ohm']:g} Ом непрерывно не менее "
    f"{activity_rule['minimum_contiguous_duration_s']:g} с."
)

print("\nСогласованность порядка файлов с протоколом 140 → 50 мм:")
for subject_id, group in record_table[
    record_table["роль"].isin(["размерный ряд", "побайтовый дубликат"])
].groupby("доброволец"):
    ordered = group.sort_values("mtime")
    observed = [int(value) for value in ordered["размер, мм"].dropna()]
    independent = ordered[ordered["решение"] == "самостоятельная запись размерного ряда"]
    independent_sizes = [int(value) for value in independent["размер, мм"]]
    expected = sorted(independent_sizes, reverse=True)
    print(
        f"  {subject_id}: порядок по mtime {observed}; "
        f"независимый ряд согласован — {'да' if independent_sizes == expected else 'нет'}"
    )


,доброволец,"размер, мм",роль,решение,"длительность, с","частота, Гц","каналы, прошедшие формальное правило",доля BASE1 выше порога,доля BASE2 выше порога,физических копий
0,Георг,50.0,размерный ряд,самостоятельная запись размерного ряда,79.42,200.0,"1, 2",1.0000,1.0,2
1,Георг,60.0,размерный ряд,самостоятельная запись размерного ряда,69.62,200.0,"1, 2",1.0000,1.0,2
2,Георг,70.0,размерный ряд,самостоятельная запись размерного ряда,71.42,200.0,"1, 2",1.0000,1.0,2
3,Георг,80.0,размерный ряд,самостоятельная запись размерного ряда,73.42,200.0,"1, 2",1.0000,1.0,2
4,Георг,90.0,размерный ряд,самостоятельная запись размерного ряда,77.62,200.0,"1, 2",0.4058,1.0,2
5,Георг,100.0,размерный ряд,самостоятельная запись размерного ряда,79.62,200.0,"1, 2",0.5766,1.0,2
6,Георг,110.0,размерный ряд,самостоятельная запись размерного ряда,79.82,200.0,"1, 2",1.0000,1.0,2
7,Георг,120.0,размерный ряд,самостоятельная запись размерного ряда,80.82,200.0,"1, 2",1.0000,1.0,2
8,Георг,130.0,размерный ряд,самостоятельная запись размерного ряда,84.02,200.0,"1, 2",1.0000,1.0,2
9,Георг,140.0,размерный ряд,самостоятельная запись размерного ряда,80.22,200.0,"1, 2",1.0000,1.0,2


,CSV с ожидаемой схемой,уникальных содержаний,CSV с другой схемой,неклассифицированных уникальных
доброволец,,,,
exp02_georg,21,11,11,0
exp02_nik,11,10,11,0


Статус манифеста: кандидатный, требуется ручная проверка.
Независимых записей размерного ряда: 19
Других сохранённых записей: 3
Удалённых записей: 0; исключённых каналов: 0.
Формальное правило канала: BASE > 5 Ом непрерывно не менее 4 с.

Согласованность порядка файлов с протоколом 140 → 50 мм:
  Георг: порядок по mtime [140, 130, 120, 110, 100, 90, 80, 70, 60, 50]; независимый ряд согласован — да
  Ник: порядок по mtime [140, 130, 120, 110, 90, 80, 70, 60, 50, 100]; независимый ряд согласован — да


## Покомпонентный технический разбор

Каждая физическая запись рассматривается отдельно. Поле «текущее
использование» задаёт роль записи в следующем этапе, а не решение удалить
данные. Оба канала сохраняются в исходном архиве и карточках. Для записей
Георга 90 и 100 мм ограничения задаются поканально: канал 1 и совместный
анализ каналов исключены, а канал 2 остаётся условным кандидатом после
дыхательной и ЭКГ-разметки. Записи целиком не исключаются.

Доли BASE выше 5 Ом, корреляция RHEO1/RHEO2, доли нулевых и предельных QS и
длительности крайних плато являются описательными показателями. Корреляция
рассчитана по полной записи и может быть завышена общими переходами или
плато. Длительное точное плато считается кандидатом на ограничение шкалы. До
проверки прибора такой интервал не используется для амплитуд и экстремумов, но
остальная запись и канал сохраняются.

Обзор ЭКГ здесь выявляет только грубые технические особенности. Пригодность
комплексов и R-зубцов принимает 11.02.


In [2]:
# @title Карточки всех измерений: наблюдения и рекомендации
import numpy as np

PLATEAU_REVIEW_THRESHOLD_S = 0.1
CHANNEL1_EXCLUDED_FROM_QUANTITATIVE_ANALYSIS = {
    "exp02_georg_090mm",
    "exp02_georg_100mm",
}
JOINT_CHANNEL_ANALYSIS_EXCLUDED = set(
    CHANNEL1_EXCLUDED_FROM_QUANTITATIVE_ANALYSIS
)


def longest_exact_run_s(time_s, values, target):
    mask = np.asarray(values) == target
    padded = np.concatenate(([False], mask, [False]))
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    if len(changes) == 0:
        return 0.0
    lengths = changes[1::2] - changes[::2]
    return float(np.max(lengths) * np.median(np.diff(time_s)))


def rheo_plateaus(time_s, values):
    result = []
    for boundary_name, boundary_value in (
        ("минимум", float(np.min(values))),
        ("максимум", float(np.max(values))),
    ):
        duration_s = longest_exact_run_s(time_s, values, boundary_value)
        if duration_s >= PLATEAU_REVIEW_THRESHOLD_S:
            result.append({
                "boundary": boundary_name,
                "value_mohm": boundary_value,
                "longest_run_s": duration_s,
            })
    return result


visual_notes = {
    "exp02_georg_090mm": (
        "На обзорном графике BASE1 скачкообразно переходит от низкого уровня "
        "к уровню около 20 Ом; запись нельзя описывать одним стабильным BASE1."
    ),
    "exp02_georg_100mm": (
        "BASE1 имеет необычно низкий и изменяющийся уровень; RHEO1 заметно "
        "отличается от RHEO2 сильнее, чем в соседних размерах."
    ),
    "exp02_georg_110mm": (
        "В заключительной части записи формы RHEO1 и RHEO2 расходятся; "
        "причина по одному графику не устанавливается."
    ),
    "exp02_nik_070mm": (
        "В средней части записи RHEO1 и RHEO2 расходятся на крупных переходах; "
        "на ЭКГ виден сдвиг базовой линии."
    ),
    "exp02_nik_120mm": (
        "Сходство RHEO1 и RHEO2 по полной записи ниже, чем у соседних размеров; "
        "это требует интервального сравнения после разметки."
    ),
    "exp02_nik_140mm": (
        "В конце записи ЭКГ имеет выраженное изменение базовой линии; "
        "этот участок должен отдельно проверить ноутбук 11.02."
    ),
    "exp02_georg_protocol_breathing": (
        "На обзорном графике различимы последовательные дыхательные режимы; "
        "RHEO1 и RHEO2 имеют близкую форму."
    ),
    "exp02_nik_protocol_breathing": (
        "На обзорном графике различимы последовательные дыхательные режимы; "
        "RHEO1 и RHEO2 имеют близкую форму."
    ),
}

review_rows = []
review_items = []
for item in RECORDS:
    source_path = (DATA_ROOT / item["relative_path"]).resolve()
    source_path.relative_to(DATA_ROOT)
    frame = pd.read_csv(source_path, encoding="utf-8")

    time_s = frame["TIME_s"].to_numpy(dtype=float)
    base1 = frame["BASE_1_Ω"].to_numpy(dtype=float)
    base2 = frame["BASE_2_Ω"].to_numpy(dtype=float)
    rheo1 = frame["RHEO_1_mΩ"].to_numpy(dtype=float)
    rheo2 = frame["RHEO_2_mΩ"].to_numpy(dtype=float)
    qs1 = frame["QS_1_Ω"].to_numpy(dtype=float)
    qs2 = frame["QS_2_Ω"].to_numpy(dtype=float)

    base1_fraction = float(np.mean(base1 > 5.0))
    base2_fraction = float(np.mean(base2 > 5.0))
    correlation = float(np.corrcoef(rheo1, rheo2)[0, 1])
    qs_zero_fraction = float(np.mean(qs1 == 0.0))
    qs_ceiling_fraction = float(np.mean(qs1 >= 4699.0))
    plateaus = {
        "RHEO1": rheo_plateaus(time_s, rheo1),
        "RHEO2": rheo_plateaus(time_s, rheo2),
    }

    channel1_excluded = (
        item["record_id"] in CHANNEL1_EXCLUDED_FROM_QUANTITATIVE_ANALYSIS
    )
    joint_analysis_excluded = item["record_id"] in JOINT_CHANNEL_ANALYSIS_EXCLUDED
    review_status = (
        "author_decision_channel_specific_quantitative_restrictions"
        if channel1_excluded
        else "pending_manual_review"
    )
    comments = []
    recommendations = [
        "Сохранить исходную запись и оба канала для прослеживаемости и QC."
    ]

    if item["role"] == "excluded_duplicate":
        comments.append("Файл побайтно совпадает с самостоятельной записью Ника 90 мм.")
        recommendations.append(
            "Хранить для прослеживаемости, но не считать вторым независимым "
            "измерением и не удваивать статистический вес."
        )
        use_label = "прослеживаемость; не независимый повтор"
    elif item.get("side_size_mm") is None:
        comments.append("Размер боковой сборки для протокольной записи не установлен.")
        recommendations.append(
            "Использовать для восстановления дыхательного протокола и "
            "сопоставления каналов; не включать в зависимость от размера до "
            "восстановления размера."
        )
        use_label = "протокол дыхания; не размерная зависимость"
    else:
        comments.append("Самостоятельная запись размерного ряда.")
        recommendations.append(
            "Передать в 11.01 и 11.02; количественные амплитуды рассчитывать "
            "только после принятия интервалов."
        )
        use_label = "кандидат размерного ряда"

    if channel1_excluded:
        comments.append(
            "Решением автора BASE1 и RHEO1 исключены из количественных расчётов. "
            "BASE2 и RHEO2 автоматически не исключаются."
        )
        recommendations.append(
            "BASE2 передать в 11.01 и 12.01 как условный кандидат для проверки "
            "на принятых задержках вдоха и выдоха. RHEO2 передать в 11.02 и "
            "использовать только по принятым сердечным циклам без предельных "
            "плато. Совместный количественный анализ каналов 1 и 2 не выполнять."
        )
        use_label = (
            "канал 1 и совместный анализ исключены; канал 2 — условный кандидат"
        )

    if base1_fraction < 0.99:
        comments.append(f"BASE1 выше 5 Ом только в {100 * base1_fraction:.2f}% записи.")
        recommendations.append(
            "Канал 1 анализировать по отдельным временным участкам и не "
            "использовать как доказанно стабильный референс."
        )
    else:
        comments.append("BASE1 выше формального порога 5 Ом на всей записи.")

    if base2_fraction < 0.99:
        comments.append(f"BASE2 выше 5 Ом только в {100 * base2_fraction:.2f}% записи.")
        recommendations.append("Канал 2 анализировать по отдельным временным участкам.")
    else:
        comments.append("BASE2 выше формального порога 5 Ом на всей записи.")

    plateau_parts = []
    for channel_name, channel_plateaus in plateaus.items():
        for plateau in channel_plateaus:
            plateau_parts.append(
                f"{channel_name}: {plateau['boundary']} "
                f"{plateau['value_mohm']:.3f} мОм, "
                f"до {plateau['longest_run_s']:.3f} с"
            )
    if plateau_parts:
        comments.append(
            "Длительные точные плато на крайних значениях: "
            + "; ".join(plateau_parts)
            + ". Это признак возможного ограничения шкалы, а не установленная причина."
        )
        recommendations.append(
            "Пометить плато как технически сомнительные интервалы и не брать "
            "из них амплитуды или экстремумы; остальные интервалы и каналы "
            "не исключать."
        )
    else:
        comments.append(
            "Плато RHEO длительностью не менее 0,1 с на точном минимуме или "
            "максимуме записи не обнаружено."
        )

    comments.append(
        f"QS равен нулю в {100 * qs_zero_fraction:.2f}% отсчётов"
        + (
            f" и не ниже 4699 Ом в {100 * qs_ceiling_fraction:.2f}% отсчётов."
            if qs_ceiling_fraction > 0
            else "; значений не ниже 4699 Ом не обнаружено."
        )
    )
    if np.array_equal(qs1, qs2):
        comments.append(
            "QS1 и QS2 полностью совпадают по значениям и не считаются двумя "
            "независимыми показателями."
        )
    comments.append(
        f"Описательная корреляция RHEO1/RHEO2 по полной записи: r={correlation:.3f}."
    )
    if correlation >= 0.9:
        recommendations.append(
            "Высокое сходство каналов не трактовать как доказательство одного "
            "источника; проверить межканальное влияние на принятых интервалах."
        )
    elif correlation < 0.6:
        recommendations.append(
            "Не переносить амплитуду одного канала на другой; после разметки "
            "сравнить их на одинаковых физиологических интервалах."
        )

    if item["record_id"] in visual_notes:
        comments.append(visual_notes[item["record_id"]])

    metrics = {
        "base1_fraction_above_5": base1_fraction,
        "base2_fraction_above_5": base2_fraction,
        "rheo1_rheo2_full_record_pearson_r": correlation,
        "qs_zero_fraction": qs_zero_fraction,
        "qs_ge_4699_fraction": qs_ceiling_fraction,
        "qs_columns_equal": bool(np.array_equal(qs1, qs2)),
        "candidate_extreme_plateaus": plateaus,
    }
    review_items.append({
        "record_id": item["record_id"],
        "input_sha256": item["input_sha256"],
        "review_status": review_status,
        "retain_record": True,
        "retain_channels": [1, 2],
        "quantitative_use": (
            "channel_specific_restrictions"
            if channel1_excluded
            else "pending_manual_review"
        ),
        "channel_quantitative_use": {
            "BASE1": "excluded" if channel1_excluded else "pending_manual_review",
            "RHEO1": "excluded" if channel1_excluded else "pending_manual_review",
            "BASE2": "pending_breathing_interval_review",
            "RHEO2": "pending_ecg_review_nonplateau_cycles_only",
            "joint_channels_1_2": (
                "excluded" if joint_analysis_excluded else "pending_manual_review"
            ),
        },
        "current_use": use_label,
        "comments": comments,
        "recommendations": recommendations,
        "metrics": metrics,
    })
    review_rows.append({
        "доброволец": SUBJECT_LABELS[item["subject_id"]],
        "размер, мм": item.get("side_size_mm"),
        "роль": role_labels.get(item["role"], item["role"]),
        "статус": review_status,
        "каналы": "1 и 2 сохранены в архиве",
        "текущее использование": use_label,
        "наблюдения": " ".join(comments),
        "рекомендации": " ".join(recommendations),
    })

record_review_table = (
    pd.DataFrame(review_rows)
    .sort_values(["доброволец", "размер, мм"], na_position="last")
    .reset_index(drop=True)
)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(record_review_table)

review_payload = {
    "schema_version": 2,
    "experiment_id": "exp02",
    "status": "pending_manual_review",
    "policy": (
        "raw_records_retained_georg_90_100_channel1_and_joint_excluded_"
        "channel2_conditional"
    ),
    "plateau_review_threshold_s": PLATEAU_REVIEW_THRESHOLD_S,
    "source_manifest": QC_MANIFEST_PATH.name,
    "items": review_items,
}
REVIEW_PATH = QC_MANIFEST_PATH.with_name("10.01_record_review.candidate.json")
if REVIEW_PATH.exists():
    existing_review = json.loads(REVIEW_PATH.read_text(encoding="utf-8"))
    if existing_review.get("status") == "accepted":
        raise RuntimeError("Принятый покомпонентный разбор нельзя перезаписывать")
temporary_review = REVIEW_PATH.with_suffix(REVIEW_PATH.suffix + ".tmp")
temporary_review.write_text(
    json.dumps(review_payload, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
temporary_review.replace(REVIEW_PATH)

plateau_flagged = sum(
    any(item["metrics"]["candidate_extreme_plateaus"].values())
    for item in review_items
)
print("Карточек измерений:", len(review_items))
print("Удалённых записей: 0; целиком исключённых записей: 0.")
print("Записей с кандидатным флагом плато RHEO:", plateau_flagged)
print(
    "Канал 1 и совместный анализ исключены для:",
    ", ".join(sorted(CHANNEL1_EXCLUDED_FROM_QUANTITATIVE_ANALYSIS)),
)
print(
    "Независимых записей — кандидатов для BASE2/RHEO2:",
    sum(
        item["role"] == "ttrkg_channel_1_with_side_channel_2_size_sweep"
        for item in RECORDS
    ),
)
print(
    "Независимых записей — кандидатов для канала 1 и совместного анализа:",
    sum(
        item["role"] == "ttrkg_channel_1_with_side_channel_2_size_sweep"
        and item["record_id"] not in CHANNEL1_EXCLUDED_FROM_QUANTITATIVE_ANALYSIS
        for item in RECORDS
    ),
)
print("Покомпонентный разбор:", REVIEW_PATH.name)


,доброволец,"размер, мм",роль,статус,каналы,текущее использование,наблюдения,рекомендации
0,Георг,50.0,размерный ряд,pending_manual_review,1 и 2 сохранены в архиве,кандидат размерного ряда,"Самостоятельная запись размерного ряда. BASE1 выше формального порога 5 Ом на всей записи. BASE2 выше формального порога 5 Ом на всей записи. Длительные точные плато на крайних значениях: RHEO1: минимум -505.199 мОм, до 1.825 с; RHEO1: максимум 584.215 мОм, до 1.305 с; RHEO2: минимум -502.814 мОм, до 2.125 с; RHEO2: максимум 502.568 мОм, до 1.425 с. Это признак возможного ограничения шкалы, а не установленная причина. QS равен нулю в 26.40% отсчётов и не ниже 4699 Ом в 4.36% отсчётов. QS1 и QS2 полностью совпадают по значениям и не считаются двумя независимыми показателями. Описательная корреляция RHEO1/RHEO2 по полной записи: r=0.725.",Сохранить исходную запись и оба канала для прослеживаемости и QC. Передать в 11.01 и 11.02; количественные амплитуды рассчитывать только после принятия интервалов. Пометить плато как технически сомнительные интервалы и не брать из них амплитуды или экстремумы; остальные интервалы и каналы не исключать.
1,Георг,60.0,размерный ряд,pending_manual_review,1 и 2 сохранены в архиве,кандидат размерного ряда,"Самостоятельная запись размерного ряда. BASE1 выше формального порога 5 Ом на всей записи. BASE2 выше формального порога 5 Ом на всей записи. Длительные точные плато на крайних значениях: RHEO1: минимум -505.199 мОм, до 1.285 с; RHEO1: максимум 584.215 мОм, до 1.345 с; RHEO2: минимум -502.814 мОм, до 2.765 с; RHEO2: максимум 502.568 мОм, до 1.625 с. Это признак возможного ограничения шкалы, а не установленная причина. QS равен нулю в 20.65% отсчётов и не ниже 4699 Ом в 2.59% отсчётов. QS1 и QS2 полностью совпадают по значениям и не считаются двумя независимыми показателями. Описательная корреляция RHEO1/RHEO2 по полной записи: r=0.812.",Сохранить исходную запись и оба канала для прослеживаемости и QC. Передать в 11.01 и 11.02; количественные амплитуды рассчитывать только после принятия интервалов. Пометить плато как технически сомнительные интервалы и не брать из них амплитуды или экстремумы; остальные интервалы и каналы не исключать.
2,Георг,70.0,размерный ряд,pending_manual_review,1 и 2 сохранены в архиве,кандидат размерного ряда,"Самостоятельная запись размерного ряда. BASE1 выше формального порога 5 Ом на всей записи. BASE2 выше формального порога 5 Ом на всей записи. Длительные точные плато на крайних значениях: RHEO1: минимум -505.199 мОм, до 0.325 с; RHEO1: максимум 584.215 мОм, до 0.225 с; RHEO2: минимум -502.814 мОм, до 1.465 с; RHEO2: максимум 502.568 мОм, до 0.705 с. Это признак возможного ограничения шкалы, а не установленная причина. QS равен нулю в 28.49% отсчётов и не ниже 4699 Ом в 1.53% отсчётов. QS1 и QS2 полностью совпадают по значениям и не считаются двумя независимыми показателями. Описательная корреляция RHEO1/RHEO2 по полной записи: r=0.798.",Сохранить исходную запись и оба канала для прослеживаемости и QC. Передать в 11.01 и 11.02; количественные амплитуды рассчитывать только после принятия интервалов. Пометить плато как технически сомнительные интервалы и не брать из них амплитуды или экстремумы; остальные интервалы и каналы не исключать.
3,Георг,80.0,размерный ряд,pending_manual_review,1 и 2 сохранены в архиве,кандидат размерного ряда,"Самостоятельная запись размерного ряда. BASE1 выше формального порога 5 Ом на всей записи. BASE2 выше формального порога 5 Ом на всей записи. Длительные точные плато на крайних значениях: RHEO1: максимум 584.215 мОм, до 1.225 с; RHEO2: минимум -502.814 мОм, до 1.985 с; RHEO2: максимум 502.568 мОм, до 1.265 с. Это признак возможного ограничения шкалы, а не установленная причина. QS равен нулю в 29.62% отсчётов и не ниже 4699 Ом в 3.60% отсчётов. QS1 и QS2 полностью совпадают по значениям и не считаются двумя независимыми показателями. Описательная корреляция RHEO1/RHEO2 по полной записи: r=0.790.",Сохранить исходную запись и оба канала для прослеживаемости

Карточек измерений: 22
Удалённых записей: 0; целиком исключённых записей: 0.
Записей с кандидатным флагом плато RHEO: 20
Канал 1 и совместный анализ исключены для: exp02_georg_090mm, exp02_georg_100mm
Независимых записей — кандидатов для BASE2/RHEO2: 19
Независимых записей — кандидатов для канала 1 и совместного анализа: 17
Покомпонентный разбор: 10.01_record_review.candidate.json


## Интерпретация результата

Машинный контракт обнаруживает 19 самостоятельных записей размерного ряда:
девять у Ника и десять у Георга. Записи Георга 90 и 100 мм целиком не
исключаются. Из-за аномального поведения `BASE1/RHEO1` канал 1 и совместный
количественный анализ каналов 1 и 2 для этих записей исключены. `BASE2`
остаётся условным кандидатом после проверки принятых задержек вдоха и выдоха,
а `RHEO2` — после ЭКГ-разметки и только по сердечным циклам без предельных
плато. Таким образом, для канала 2 сохраняются 19 независимых кандидатов
размерного ряда, а для канала 1 и совместного анализа — 17.

Файл Ника 100 мм также не удаляется, но хранится как побайтовая копия 90 мм и
не считается вторым независимым повтором. Две записи полного дыхательного
протокола сохраняются, но не участвуют в зависимости от размера, пока размер
боковой сборки не установлен.

Основной технический результат обзорного прохода — повторяющиеся длительные
плато RHEO на одинаковых крайних значениях. Пока это кандидатный признак
ограничения шкалы. Причина и допустимый диапазон не установлены. Поэтому в
будущих расчётах маркируются интервалы плато, а не исключается вся запись или
весь канал.

Необычные значения BASE и QS также являются указателями для ручного просмотра
и сами по себе не доказывают нарушение контакта или неисправность прибора.


## Передача результата

Файл 10.01_record_manifest.candidate.json записывается во внешний каталог
производных данных и содержит машинный состав записей без исходных сигналов.

Файл 10.01_record_review.candidate.json содержит карточку каждого измерения:
технические наблюдения, рекомендации, сохранение обоих каналов и статус
pending_manual_review. Он не содержит исходных сигналов или абсолютных путей.

После ручного принятия состава записей манифест передаётся в 11.01 и 11.02.
Сравнение сигналов на принятых физиологических интервалах выполняется в 12.01;
физические модели используют только последующие результаты с явно указанными
допущениями и калибровкой.
